In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split, Subset
import time
import matplotlib.pyplot as plt
import os
from sklearn.model_selection import train_test_split

import torch
import pyro
import pyro.distributions as dist
from pyro.nn.module import PyroModule, PyroParam
from pyro.infer.autoguide import AutoGuide
from pyro.infer.autoguide.initialization import InitMessenger, init_to_feasible
from pyro.distributions import constraints
from contextlib import ExitStack

import pickle
from tqdm import tqdm
import copy

import pyro
import pyro.distributions as dist
from pyro.nn import PyroModule, PyroSample
from pyro.infer.autoguide import AutoNormal

import pandas as pd

import numpy as np
from sklearn.metrics import confusion_matrix

from bitflip import bitflip_float32

from torchvision.datasets import ImageFolder

import os

import json

import argparse

from dotenv import load_dotenv
import requests



def send_telegram_message(title, message):
    load_dotenv('.env')
    token = os.getenv('TELEGRAM_BOT_TOKEN')

    try:
        response = requests.post(f'https://api.telegram.org/bot{token}/sendMessage', data={
            'chat_id': os.getenv('TELEGRAM_CHAT_ID'),
            'text': f'{title}\n{message}',
            #'parse_mode': 'Markdown'
        })
    except requests.exceptions.RequestException as e:
        print(f"Error sending message: {e}")
        return None

shipsnet_mean = [0.4119, 0.4243, 0.3724]
shipsnet_std = [0.1899, 0.1569, 0.1515]

def load_data(batch_size=16):
    transform = transforms.Compose([
        transforms.Resize((64, 64)),
        transforms.ToTensor(),
        transforms.Normalize(mean=shipsnet_mean, 
                             std=shipsnet_std)
    ])

    #dataset = datasets.EuroSAT(root='./data', transform=transform, download=True)
    dataset = ImageFolder(
    root="data/shipsnet/foldered",
    transform=transform
    )
    torch.manual_seed(42)

    #train_size = int(0.8 * len(dataset))
    #test_size = len(dataset) - train_size
    #train_dataset, test_dataset = random_split(dataset, [train_size, test_size])
    
    with open('datasplit/shipsnet_split_indices.pkl', 'rb') as f:
        split = pickle.load(f)
        train_dataset = Subset(dataset, split['train'])
        test_dataset = Subset(dataset, split['test'])

    # Add num_workers and pin_memory for faster data loading
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, 
                             num_workers=4, pin_memory=True, persistent_workers=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size,
                            num_workers=4, pin_memory=True, persistent_workers=True)
    return train_loader, test_loader

from pyro.distributions.util       import sum_rightmost
from pyro.ops.tensor_utils         import periodic_repeat
from pyro.distributions.transforms import biject_to
from pyro.infer.autoguide.utils    import (
    deep_setattr,
    deep_getattr,
    helpful_support_errors,
)

import pyro.poutine as poutine

from pyro.infer.autoguide import AutoGuideList  #, AutoLowRankMultivariateNormal\
from pyro.infer.autoguide import AutoLowRankMultivariateNormal

c:\Users\Revalda Putawara\.conda\envs\bnntest\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Added
class UniformReal(dist.Uniform):
    @property
    def support(self):
        return constraints.real

In [ ]:
class AutoUniform(AutoGuide):
    """
    An AutoGuide that uses a Uniform(low, low+width) marginal for each latent.
    """
    # `width` must be positive
    width_constraint = constraints.softplus_positive

    def __init__(
        self, model, *, init_loc_fn=init_to_feasible, init_scale=0.1, create_plates=None
    ):
        self.init_loc_fn = init_loc_fn
        if not isinstance(init_scale, float) or not (init_scale > 0):
            raise ValueError(f"Expected init_scale > 0, got {init_scale}")
        self._init_scale = init_scale

        model = InitMessenger(self.init_loc_fn)(model)
        super().__init__(model, create_plates=create_plates)

    def _setup_prototype(self, *args, **kwargs):
        super()._setup_prototype(*args, **kwargs)
        self._event_dims = {}
        self.lows = PyroModule()
        self.widths = PyroModule()

        for name, site in self.prototype_trace.iter_stochastic_nodes():
            # 1. get an unconstrained init_loc (inverse‐transform of site["value"])
            with helpful_support_errors(site):
                init_loc = (
                    biject_to(site["fn"].support)
                    .inv(site["value"].detach())
                    .detach()
                )
            event_dim = site["fn"].event_dim + init_loc.dim() - site["value"].dim()
            self._event_dims[name] = event_dim

            # 2. if subsampled, expand back to full size
            for frame in site["cond_indep_stack"]:
                full_size = frame.full_size or frame.size
                if full_size != frame.size:
                    dim = frame.dim - event_dim
                    init_loc = periodic_repeat(init_loc, full_size, dim).contiguous()

            # 3. build initial low & width around that init_loc
            init_low   = init_loc - self._init_scale
            init_width = torch.full_like(init_loc, 2.0 * self._init_scale)

            # 4. register as PyroParams
            deep_setattr(
                self.lows,  name,
                PyroParam(init_low,   constraints.real,             event_dim),
            )
            deep_setattr(
                self.widths, name,
                PyroParam(init_width, self.width_constraint,       event_dim),
            )

    def _get_low_and_width(self, name):
        low   = deep_getattr(self.lows,  name)
        width = deep_getattr(self.widths, name)
        return low, width

    def forward(self, *args, **kwargs):
        if self.prototype_trace is None:
            self._setup_prototype(*args, **kwargs)

        plates = self._create_plates(*args, **kwargs)
        result = {}

        for name, site in self.prototype_trace.iter_stochastic_nodes():
            transform = biject_to(site["fn"].support)
            with ExitStack() as stack:
                for frame in site["cond_indep_stack"]:
                    if frame.vectorized:
                        stack.enter_context(plates[frame.name])

                low, width = self._get_low_and_width(name)
                # draw unconstrained latent from Uniform(low, low + width)
                unconstrained = pyro.sample(
                    f"{name}_unconstrained",
                    #dist.Uniform(low, low + width).to_event(self._event_dims[name]),
                    UniformReal(low, low + width).to_event(self._event_dims[name]),
                    infer={"is_auxiliary": True},
                )

                # map into constrained space
                value = transform(unconstrained)
                if poutine.get_mask() is False:
                    log_density = 0.0
                else:
                    log_density = transform.inv.log_abs_det_jacobian(
                        value, unconstrained
                    )
                    log_density = sum_rightmost(
                        log_density,
                        log_density.dim() - value.dim() + site["fn"].event_dim,
                    )
                delta = dist.Delta(
                    value,
                    log_density=log_density,
                    event_dim=site["fn"].event_dim,
                )
                result[name] = pyro.sample(name, delta)

        return result

    @torch.no_grad()
    def median(self, *args, **kwargs):
        """
        Posterior median is just the 0.5‐quantile of Uniform = low + 0.5*width
        """
        medians = {}
        for name, site in self.prototype_trace.iter_stochastic_nodes():
            low, width = self._get_low_and_width(name)
            med = biject_to(site["fn"].support)(low + 0.5 * width)
            medians[name] = med.clone() if med is low else med
        return medians

    @torch.no_grad()
    def quantiles(self, quantiles, *args, **kwargs):
        """
        Posterior quantiles via Uniform.icdf(q).
        """
        results = {}
        qs = torch.tensor(quantiles)
        for name, site in self.prototype_trace.iter_stochastic_nodes():
            low, width = self._get_low_and_width(name)
            # shape: [len(quantiles), *low.shape]
            #qvals = dist.Uniform(low, low + width).icdf(qs.reshape((-1,) + (1,) * low.dim()))
            qvals = UniformReal(low, low + width).icdf(qs.reshape((-1,) + (1,) * low.dim()))
            results[name] = biject_to(site["fn"].support)(qvals)
        return results

In [ ]:
class BayesShipsCNN(PyroModule):
    def __init__(
        self,
        num_classes=2,   # now 2 for Categorical
        device=torch.device("cuda"),
        activation='relu',
        prior_dist='gaussian',
        mu=0.0,
        b=1.0,
        prior_params=None
    ):
        super().__init__()
        self.device = device

        # Activation setup
        if isinstance(activation, str):
            act_map = {
                'relu': F.relu,
                'tanh': torch.tanh,
                'sigmoid': torch.sigmoid,
                'sin': torch.sin,
                'relu6': F.relu6,
                'leaky_relu': F.leaky_relu,
                'selu': F.selu,
                'actWG': self.actWG,
                'actRWG': self.actRWG,
            }
            self.activation_fn = act_map[activation]
        elif callable(activation):
            self.activation_fn = activation
        else:
            raise ValueError("activation must be a string or callable")

        # Prior setup
        self.prior_dist = prior_dist
        params = {'mu': mu, 'b': b} if prior_params is None else prior_params
        self.prior_mu = torch.tensor(params['mu'], device=device, dtype=torch.float32)
        self.prior_b  = torch.tensor(params['b'], device=device, dtype=torch.float32)

        print(f"[INFO] Using prior: {self.prior_dist} (mu={self.prior_mu.item()}, b={self.prior_b.item()})")

        # Layers
        self.conv1 = PyroModule[nn.Conv2d](3, 32, kernel_size=3, padding=1)
        self.conv1.weight = PyroSample(self._make_prior([32, 3, 3, 3]))
        self.conv1.bias   = PyroSample(self._make_prior([32]))

        self.conv2 = PyroModule[nn.Conv2d](32, 64, kernel_size=3, padding=1)
        self.conv2.weight = PyroSample(self._make_prior([64, 32, 3, 3]))
        self.conv2.bias   = PyroSample(self._make_prior([64]))

        self.pool = nn.MaxPool2d(2, 2)

        self.fc1 = PyroModule[nn.Linear](64 * 16 * 16, num_classes)
        self.fc1.weight = PyroSample(self._make_prior([num_classes, 64 * 16 * 16]))
        self.fc1.bias   = PyroSample(self._make_prior([num_classes]))

    def actWG(self, x, alpha=1.0):
        return x * torch.exp(-alpha * x ** 2)

    def actRWG(self, x, alpha=1.0):
        wg = x * torch.exp(-alpha * x ** 2)
        return torch.max(torch.zeros_like(wg), wg)

    def _make_prior(self, shape):
        if self.prior_dist == 'gaussian':
            base = dist.Normal(self.prior_mu, self.prior_b)
        elif self.prior_dist == 'laplace':
            base = dist.Laplace(self.prior_mu, self.prior_b)
        elif self.prior_dist == 'uniform':
            #base = dist.Uniform(-self.prior_b, self.prior_b)
            base = UniformReal(-self.prior_b, self.prior_b)
        else:
            raise ValueError(f"Unsupported prior: {self.prior_dist}")
        return base.expand(shape).to_event(len(shape))

    def forward(self, x, y=None):
        x = self.activation_fn(self.conv1(x))
        x = self.pool(x)
        x = self.activation_fn(self.conv2(x))
        x = self.pool(x)

        x = x.view(x.size(0), -1)
        logits = self.fc1(x)  # shape [batch, 2]

        if y is not None:
            with pyro.plate("data", x.size(0)):
                pyro.sample("obs", dist.Categorical(logits=logits), obs=y)
        return logits


def load_model(timestamp):
    config_path = os.path.join(search_dir, config_files[timestamp])
    guide_path = os.path.join(search_dir, guide_files[timestamp])
    model_path = os.path.join(search_dir, model_files[timestamp])
    param_path = os.path.join(search_dir, param_files[timestamp])

    print(f"Loading model with config_path: {config_path}")

    with open(config_path, 'r') as f:
        config = json.load(f)

    model = BayesShipsCNN(
        num_classes=num_classes,
        device=device,
        activation=config['activation'],
        prior_dist=config['prior'],
        mu=config['prior_params']['mu'],
        b=config['prior_params'],
        prior_params=config.get('prior_params', None)
    ).to(device)

    # Load the guide
    #guide = AutoDiagonalNormal(model)

    # Load the model state
    model.load_state_dict(torch.load(model_path))
    
    # Load the guide state
    #guide.load_state_dict(torch.load(guide_path))

    return model, param_path

class NewInjector:
    def __init__(self, trained_model, device, test_loader, num_samples, multivariate_flag=False):
        """
        Initializes SEU injector
        """
        self.device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.trained_model = trained_model.to(self.device)
        self.test_loader = test_loader
        self.trained_model.eval()
        self.num_samples = num_samples

        #self.guide = AutoDiagonalNormal(self.trained_model).to(self.device)
        if self.trained_model.prior_dist == 'gaussian':
            if multivariate_flag:
                self.guide = AutoGuideList(bayesian_model)

                # 1) conv1.weight
                self.guide.add(
                    AutoLowRankMultivariateNormal(
                        poutine.block(bayesian_model, expose=["conv1.weight"]),
                        rank=20,
                        init_scale=0.05,
                        #prefix="AutoGuideList.conv1.weight"
                    )
                )

                # 2) conv1.bias
                self.guide.add(
                    AutoLowRankMultivariateNormal(
                        poutine.block(bayesian_model, expose=["conv1.bias"]),
                        rank=5,
                        init_scale=0.05,
                        #prefix="AutoGuideList.conv1.bias"
                    )
                )

                # 3) conv2.weight
                self.guide.add(
                    AutoLowRankMultivariateNormal(
                        poutine.block(bayesian_model, expose=["conv2.weight"]),
                        rank=20,
                        init_scale=0.05,
                        #prefix="AutoGuideList.conv2.weight"
                    )
                )

                # 4) conv2.bias
                self.guide.add(
                    AutoLowRankMultivariateNormal(
                        poutine.block(bayesian_model, expose=["conv2.bias"]),
                        rank=5,
                        init_scale=0.05,
                        #prefix="AutoGuideList.conv2.bias"
                    )
                )

                # 5) fc1.weight
                self.guide.add(
                    AutoLowRankMultivariateNormal(
                        poutine.block(bayesian_model, expose=["fc1.weight"]),
                        rank=20,
                        init_scale=0.05,
                        #prefix="AutoGuideList.fc1.weight"
                    )
                )

                # 6) fc1.bias
                self.guide.add(
                    AutoLowRankMultivariateNormal(
                        poutine.block(bayesian_model, expose=["fc1.bias"]),
                        rank=5,
                        init_scale=0.05,
                        #prefix="AutoGuideList.fc1.bias"
                    )
                )
            else:
                self.guide = AutoNormal(self.trained_model, init_scale=0.05).to(self.device)
        elif self.trained_model.prior_dist == 'laplace':
            self.guide = AutoLaplace(self.trained_model, init_scale=0.05).to(self.device)
        elif self.trained_model.prior_dist == 'uniform':
            self.guide = AutoUniform(self.trained_model, init_scale=0.05).to(self.device)
        else:
            raise ValueError(f"Unsupported prior: {self.trained_model.prior_dist}")
        
        pyro.get_param_store().clear()
        pyro.get_param_store().set_state(torch.load(pyro_param_store_path, weights_only=False))

        initial_labels, initial_predictions, initial_logits, initial_probs = self.predict_data_probs(self.num_samples)
        self.initial_accuracy = self.return_accuracy(initial_labels, initial_predictions)
        self.initial_probs = np.array(initial_probs)

        print(f"Initial accuracy: {self.initial_accuracy:.3%}")

    def predict_data_probs(self, num_samples=10):
        all_labels = []
        all_predictions = []
        all_logits = []
        all_probs = []

        with torch.no_grad():
            for images, labels in tqdm(self.test_loader, desc="Evaluating"):
                images, labels = images.to(self.device), labels.to(self.device)
                logits_mc = torch.zeros(num_samples, images.size(0), self.trained_model.fc1.out_features).to(self.device)

                # ADDED
                for i in range(num_samples):
                    guide_trace = pyro.poutine.trace(self.guide).get_trace(images)
                #    if i == 0:  # Only print once per batch
                #        # Extract 'conv1.weight' sample site from the guide trace
                #        if "conv1.weight" in guide_trace.nodes:
                #            conv1_weight = guide_trace.nodes["conv1.weight"]["value"]
                #            print(f"'conv1.weight' sampled value (batch {images.shape[0]}):")
                #            print(conv1_weight)
                #        else:
                #            print("'conv1.weight' not found in the guide trace.")
                # END OF ADDED
                    
                    replayed_model = pyro.poutine.replay(self.trained_model, trace=guide_trace)
                    logits = replayed_model(images)
                    logits_mc[i] = logits

                avg_logits = logits_mc.mean(dim=0)
                predictions = torch.argmax(avg_logits, dim=1)

                all_labels.extend(labels.cpu().numpy())
                all_predictions.extend(predictions.cpu().numpy())
                all_logits.extend(avg_logits.cpu().numpy())
                all_probs.extend(F.softmax(avg_logits, dim=1).cpu().numpy())

        return all_labels, all_predictions, all_logits, all_probs

    def return_accuracy(self, all_labels, all_predictions):
        cm = confusion_matrix(all_labels, all_predictions)
        return np.trace(cm) / np.sum(cm)

    def compute_softmax_difference(self, before_probs, after_logits, penalty=1.0):
        """
        before_probs: list or array, shape (N, C), all finite probabilities
        after_logits: list or array, shape (N, C), raw logits (may contain ±inf)
        penalty: float, the per‑example penalty to use if logits are nonfinite
        
        Returns the mean over N examples of either
        - max_i |before_probs[n,i] − after_probs[n,i]|,  if after_logits[n] is finite
        - penalty,                                    otherwise
        """
        before = np.asarray(before_probs, dtype=np.float32)
        after_logits = torch.from_numpy(np.asarray(after_logits, dtype=np.float32))
        N, C = after_logits.shape

        # 1) detect which rows of after_logits are all finite
        finite_mask = torch.isfinite(after_logits).all(dim=1).numpy()  # shape (N,)

        # 2) safe‑softmax only on the finite ones
        safe_after_probs = torch.zeros_like(after_logits)
        if finite_mask.any():
            good_logits = after_logits[finite_mask]
            # (you can optionally do the “stable” shift here)
            safe_after_probs[finite_mask] = F.softmax(good_logits, dim=1)
        safe_after_probs = safe_after_probs.numpy()

        # 3) compute per‑example diff, using the penalty where needed
        diffs = np.empty(N, dtype=np.float32)
        for n in range(N):
            if not finite_mask[n]:
                diffs[n] = penalty
            else:
                diffs[n] = np.max(np.abs(before[n] - safe_after_probs[n]))
        return diffs.mean()

    def compute_difference(self, original_val, modified_val):
        return abs(original_val - modified_val)
    
    def run_seu_multivariate(self, location_index, parameter_name, ll_module_index, bit_i, num_samples):
        assert parameter_name in ["loc", "scale"]
        
        pyro.get_param_store().set_state(torch.load(pyro_param_store_path, weights_only=False))

        #param_store_name_initial = f"{param_unique}.{parameter_name}.{layer}.{layer_module}"
        #TODO translate the param_store_name
        # turn things like AutoNormal.locs.conv1.weight into  AutoGuideList.0.loc
        # 0 is conv1.weight, 1 is conv1.bias, etc

        #if f"{layer}.{layer_module}" == "conv1.weight":
        #    layer_module_number = "0"
        #elif f"{layer}.{layer_module}" == "conv1.bias":
        #    layer_module_number = "1"
        #elif f"{layer}.{layer_module}" == "conv2.weight":
        #    layer_module_number = "2"
        #elif f"{layer}.{layer_module}" == "conv2.bias":
        #    layer_module_number = "3"
        #elif f"{layer}.{layer_module}" == "fc1.weight":
        #    layer_module_number = "4"
        #elif f"{layer}.{layer_module}" == "fc1.bias":
        #    layer_module_number = "5"

        #print(f"{layer}.{layer_module}")

        param_store_name = f"AutoGuideList.{ll_module_index}.{parameter_name}"

        with torch.no_grad():
            param = pyro.get_param_store().get_param(param_store_name)
            new_param = param.clone()
            new_param = new_param.view(-1) #flatten new param
            original_val = new_param[location_index].cpu().item()
            seu_val = bitflip_float32(original_val, bit_i)
            abs_diff = self.compute_difference(original_val, seu_val)
            new_param[location_index] = seu_val
            # return new_param to original shape
            new_param = new_param.view(param.shape)
            pyro.get_param_store().__setitem__(param_store_name, new_param)

            print(f"Original value: {original_val}, SEU value: {seu_val}, Abs difference: {abs_diff}")

        guide = AutoGuideList(bayesian_model)

        # 1) conv1.weight
        guide.add(
            AutoLowRankMultivariateNormal(
                poutine.block(bayesian_model, expose=["conv1.weight"]),
                rank=20,
                init_scale=0.05,
                #prefix="AutoGuideList.conv1.weight"
            )
        )

        # 2) conv1.bias
        guide.add(
            AutoLowRankMultivariateNormal(
                poutine.block(bayesian_model, expose=["conv1.bias"]),
                rank=5,
                init_scale=0.05,
                #prefix="AutoGuideList.conv1.bias"
            )
        )

        # 3) conv2.weight
        guide.add(
            AutoLowRankMultivariateNormal(
                poutine.block(bayesian_model, expose=["conv2.weight"]),
                rank=20,
                init_scale=0.05,
                #prefix="AutoGuideList.conv2.weight"
            )
        )

        # 4) conv2.bias
        guide.add(
            AutoLowRankMultivariateNormal(
                poutine.block(bayesian_model, expose=["conv2.bias"]),
                rank=5,
                init_scale=0.05,
                #prefix="AutoGuideList.conv2.bias"
            )
        )

        # 5) fc1.weight
        guide.add(
            AutoLowRankMultivariateNormal(
                poutine.block(bayesian_model, expose=["fc1.weight"]),
                rank=20,
                init_scale=0.05,
                #prefix="AutoGuideList.fc1.weight"
            )
        )

        # 6) fc1.bias
        guide.add(
            AutoLowRankMultivariateNormal(
                poutine.block(bayesian_model, expose=["fc1.bias"]),
                rank=5,
                init_scale=0.05,
                #prefix="AutoGuideList.fc1.bias"
            )
        )

        try:
            after_labels, after_predictions, after_logits, after_probs = self.predict_data_probs(num_samples)
            accuracy_after = self.return_accuracy(after_labels, after_predictions)
            softmax_diff = self.compute_softmax_difference(self.initial_probs, after_probs)
        except:
            print("Error during prediction after SEU.")
            accuracy_after = np.nan
            softmax_diff = np.nan

        print(f"Accuracy after SEU: {accuracy_after}")
        print("===================================")

        return {
            "accuracy_change": accuracy_after - self.initial_accuracy,
            "softmax_difference": softmax_diff,
            "absolute_difference": abs_diff
        }
    
    def run_seu_old(self, location_index, param_unique, parameter_name, layer, layer_module, bit_i, num_samples):
        assert parameter_name in ["locs", "scales", "lows", "widths"], "Parameter name must be 'locs' or 'scales'."
        assert bit_i in range(0, 33), "Bit index must be between 0 and 32."

        param_store_name = f"{param_unique}.{parameter_name}.{layer}.{layer_module}"
        pyro.get_param_store().set_state(torch.load(pyro_param_store_path, weights_only=False))

        with torch.no_grad():
            param = pyro.get_param_store().get_param(param_store_name)
            new_param = param.clone()
            new_param = new_param.view(-1) #flatten new param
            original_val = new_param[location_index].cpu().item()
            seu_val = bitflip_float32(original_val, bit_i)
            abs_diff = self.compute_difference(original_val, seu_val)
            new_param[location_index] = seu_val
            # return new_param to original shape
            new_param = new_param.view(param.shape)
            pyro.get_param_store().__setitem__(param_store_name, new_param)

            print(f"Original value: {original_val}, SEU value: {seu_val}, Abs difference: {abs_diff}")


        if param_unique == "AutoNormal":
            self.guide = AutoNormal(self.trained_model, init_scale=0.05).to(self.device)
        elif param_unique == "AutoLaplace":
            self.guide = AutoLaplace(self.trained_model, init_scale=0.05).to(self.device)
        elif param_unique == "AutoUniform":
            self.guide = AutoUniform(self.trained_model, init_scale=0.05).to(self.device)
        else:
            raise ValueError(f"Unsupported parameter unique: {param_unique}")

        #after_labels, after_predictions, after_logits, after_probs = self.predict_data_probs(num_samples)
        #accuracy_after = self.return_accuracy(after_labels, after_predictions)
        #softmax_diff = self.compute_softmax_difference(self.initial_probs, after_probs)

        try:
            after_labels, after_predictions, after_logits, after_probs = self.predict_data_probs(num_samples)
            accuracy_after = self.return_accuracy(after_labels, after_predictions)
            softmax_diff = self.compute_softmax_difference(self.initial_probs, after_probs)
        except:
            print("Error during prediction after SEU.")
            accuracy_after = np.nan
            softmax_diff = np.nan

        print(f"Accuracy after SEU: {accuracy_after}")
        print("===================================")

        return {
            "accuracy_change": accuracy_after - self.initial_accuracy,
            "softmax_difference": softmax_diff,
            "absolute_difference": abs_diff
        }


    #AutoNormal.locs.conv1.weight
    def run_seu(
        self,
        location_index: int,
        param_unique: str,
        parameter_name: str,
        layer: str,
        layer_module: str,
        bit_i: int,
        num_samples: int,
    ):
        assert parameter_name in ["locs", "scales", "lows", "widths"], \
            "Parameter name must be one of 'locs', 'scales', 'lows', or 'widths'."
        assert 0 <= bit_i < 32, "Bit index must be between 0 and 31."

        # Construct the Pyro ParamStore key for this tensor
        param_store_name = f"{param_unique}.{parameter_name}.{layer}.{layer_module}"

        # Reload the saved ParamStore so we start from your trained guide
        pyro.get_param_store().set_state(
            torch.load(pyro_param_store_path, weights_only=False)
        )

        with torch.no_grad():
            # 1) Grab the original tensor and flatten it
            param = pyro.get_param_store().get_param(param_store_name)
            flat  = param.clone().view(-1)

            # 2) Extract, flip one bit, and wrap back as a tensor
            orig_val = flat[location_index].cpu().item()
            flipped  = bitflip_float32(orig_val, bit_i)      # Python float
            seu_val  = torch.tensor(
                flipped, dtype=param.dtype, device=param.device
            )
            abs_diff = self.compute_difference(orig_val, flipped)

            # 3) Write the flipped value into the ParamStore
            flat[location_index] = seu_val
            pyro.get_param_store().__setitem__(
                param_store_name, flat.view(param.shape)
            )

            # 4) If this is an AutoUniform guide, enforce
            #    width >= nextafter(low) – low so Uniform(low, low+width) stays valid.
            if param_unique == "AutoUniform":
                # (a) get the current lows tensor (after any flip)
                low_name = f"{param_unique}.lows.{layer}.{layer_module}"
                low_param = pyro.get_param_store() \
                                .get_param(low_name) \
                                .view(-1)

                # determine the 'low' value we should use at this index:
                if parameter_name == "lows":
                    low_val = seu_val
                else:  # we just flipped a width, so low stays original
                    low_val = low_param[location_index]

                # compute the smallest positive increment (ULP) at low_val
                delta = 10 * (
                    torch.nextafter(
                        low_val,
                        torch.tensor(float("inf"), dtype=low_val.dtype, device=low_val.device),
                    )
                    - low_val
                )

                # (b) now clamp the corresponding width
                width_name  = f"{param_unique}.widths.{layer}.{layer_module}"
                width_param = pyro.get_param_store().get_param(width_name)
                wflat       = width_param.clone().view(-1)

                # original width at that index (after any SEU if param was 'widths')
                orig_width = wflat[location_index].cpu().item()
                # enforce the minimum
                new_width  = torch.max(wflat[location_index], delta)
                wflat[location_index] = new_width

                # write back the clamped widths
                pyro.get_param_store().__setitem__(
                    width_name, wflat.view(width_param.shape)
                )

                print(
                    f"Adjusted width at '{width_name}[{location_index}]': "
                    f"{orig_width:.3e} → {new_width:.3e}"
                )

            # 5) Report the flip
            print(
                f"Parameter '{param_store_name}[{location_index}]': "
                f"{orig_val:.6g} → {flipped:.6g}  "
                f"(abs diff {abs_diff:.3g})"
            )

        # 6) Re‑instantiate your guide so Predictive will pick up the perturbed params
        #if param_unique == "AutoNormal":
        #    self.guide = AutoNormal(self.trained_model, init_scale=0.05).to(self.device)
        #elif param_unique == "AutoLaplace":
        #    self.guide = AutoLaplace(self.trained_model, init_scale=0.05).to(self.device)
        #elif param_unique == "AutoUniform":
        #    self.guide = AutoUniform(self.trained_model, init_scale=0.05).to(self.device)
        #else:
        #    raise ValueError(f"Unsupported guide type: {param_unique}")

        # 7) Re‑run inference & evaluation
        after_labels, after_preds, after_logits, after_probs = self.predict_data_probs(num_samples)
        accuracy_after = self.return_accuracy(after_labels, after_preds)
        softmax_diff  = self.compute_softmax_difference(self.initial_probs, after_probs)

        print(f"Accuracy after SEU: {accuracy_after:.3%}")
        print("===================================")

        return {
            "accuracy_change":    accuracy_after - self.initial_accuracy,
            "softmax_difference": softmax_diff,
            "absolute_difference": abs_diff,
        }


    def run_seu_autodiagonal_normal_multi(self, location_indices, bit_i, parameter_name="loc",
                                          attack_ratio=1.0, num_samples=10, seed=None):
        assert parameter_name in ["loc", "scale"], "Parameter name must be 'loc' or 'scale'."
        assert bit_i in range(0, 33), "Bit index must be between 0 and 32."
        assert 0.0 <= attack_ratio <= 1.0, "Attack ratio must be between 0.0 and 1.0."

        if isinstance(location_indices, int):
            location_indices = [location_indices]

        if seed is not None:
            np.random.seed(seed)
            torch.manual_seed(seed)

        num_attacks = max(1, int(len(location_indices) * attack_ratio))
        attack_locations = np.random.choice(location_indices, size=num_attacks, replace=False)
        param_store_name = f"AutoDiagonalNormal.{parameter_name}"
        pyro.get_param_store().set_state(torch.load(pyro_param_store_path, weights_only=False))

        abs_differences = []

        with torch.no_grad():
            param = pyro.get_param_store().get_param(param_store_name)
            new_param = param.clone()

            #print(f"Attacking {num_attacks} out of {len(location_indices)} locations:")

            for location_index in attack_locations:
                original_val = new_param[location_index].cpu().item()
                seu_val = bitflip_float32(original_val, bit_i)
                abs_diff = self.compute_difference(original_val, seu_val)
                abs_differences.append(abs_diff)
                new_param[location_index] = seu_val
                print(f"  Location {location_index}: {original_val} -> {seu_val}, Log diff: {abs_diff}")

            pyro.get_param_store().__setitem__(param_store_name, new_param)

        self.guide = AutoDiagonalNormal(self.trained_model).to(self.device)

        try:
            after_labels, after_predictions, after_logits, after_probs = self.predict_data_probs(num_samples)
            accuracy_after = self.return_accuracy(after_labels, after_predictions)
            softmax_diff = self.compute_softmax_difference(self.initial_probs, after_probs)
            mean_abs_diff = float(np.mean(abs_differences))
        except:
            accuracy_after = np.nan
            softmax_diff = np.nan
            mean_abs_diff = np.nan

        #print(f"Accuracy after SEU: {accuracy_after}")
        #print("===================================")

        return {
            "accuracy_change": accuracy_after - self.initial_accuracy,
            "softmax_difference": softmax_diff,
            "mean_abs_difference": mean_abs_diff
        }

def load_model_config(timestamp):
    config_path = os.path.join(search_dir, config_files[timestamp])

    with open(config_path, 'r') as f:
        model_config = json.load(f)

    return model_config
        

In [ ]:
train_loader, test_loader = load_data(batch_size=16)
device = torch.device("cuda")
num_classes = 2

#save_dir = os.path.join("results_4_variants","results_GP_shipsnet_newslate_guide_SEU")
#search_dir = os.path.join("results_4_variants","results_GP_shipsnet_newslate_guide")

#save_dir = os.path.join("results_4_variants","results_shipsnet_03_SEU")
#search_dir = os.path.join("results_4_variants","results_shipsnet_03")

search_dir = "results_shipsnet_uniform_01"

#list all .json files in the directory
all_files = [f for f in os.listdir(search_dir)]
json_files = [f for f in os.listdir(search_dir) if f.endswith('.json')]

# excluding the format, get the last 16 characters of each filename
timestamps = [f[:-5][-16:] for f in json_files]
print("Timestamps found count:", len(timestamps))

# remove some timestamps that are not needed
# those are the ones that are not in the shipsnet_seu_result directory, without the .csv extension
#excluded_timestamps = [f[:-4][-16:] for f in [f for f in os.listdir(save_dir) if f.endswith('.csv')]]
#excluded_timestamps = []

#timestamps = [ts for ts in timestamps if ts not in excluded_timestamps]
#timestamps = timestamps[:1]
#print("After excluding, timestamps count:", len(timestamps))

# for each timestamp, look for every other files in the directory that contains the timestamp

config_files = {}
guide_files = {}
model_files = {}
param_files = {}

for timestamp in timestamps:
    config_files[timestamp] = [f for f in all_files if timestamp in f and f.endswith('.json')][0]
    guide_files[timestamp] = [f for f in all_files if timestamp in f and f.startswith('guide')][0]
    model_files[timestamp] = [f for f in all_files if timestamp in f and f.startswith('model')][0]
    param_files[timestamp] = [f for f in all_files if timestamp in f and f.startswith('param')][0]

# create a list that maps timestamps and the output of load_model_config['prior']
prior_list = []
for ts in timestamps:
    model_config = load_model_config(ts)
    prior_list.append((ts, model_config['prior']))

In [ ]:
timestamp_target = timestamps[0]

In [ ]:
target_model_config = load_model_config(timestamp_target)
target_model_config

In [ ]:
prior_guide_map = {
    'gaussian': 'AutoNormal',
    'laplace':'AutoLaplace',
    'uniform': 'AutoUniform'
}

prior_parameter_map = {
    'gaussian': 'locs',
    'laplace':'locs',
    'uniform': 'lows'
}

target_index = 0
param_unique_target = prior_guide_map[target_model_config['prior']]
parameter_name = prior_parameter_map[target_model_config['prior']]
layer_list_iter = "conv1"
module_iter = "weight"
bit_iter = 1

ts_idx = 0
#timestamp_target = timestamps[ts_idx]


pyro.clear_param_store()


bayesian_model, pyro_param_store_path = load_model(timestamp_target)

newinj = NewInjector(trained_model=bayesian_model, device=device, test_loader=test_loader, num_samples=10)

result = newinj.run_seu(target_index, param_unique_target, parameter_name, layer_list_iter, module_iter, bit_iter, num_samples=10)